<a href="https://colab.research.google.com/github/Kaushik-Donthula/AIML_insem/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [ ]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

## Lex only

In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include <stdio.h>
    int ctChar=0;
    int ctSpace=0;
    int ctWord=0;
    int ctLine=0;
%}
WORD [^ \t\n,\.:]+
EOL [\n]
BLANK [ ]
%%

{WORD} {ctWord++; ctChar+=yyleng;}
{BLANK} {ctSpace++;}
{EOL} {ctLine++;}
. {ctChar++;}
%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    yylex();

    printf("Word Count: %d\n",ctWord);
    printf("Character Count: %d\n",ctChar);
    printf("Space Count: %d\n",ctSpace);
    printf("Line Count: %d\n",ctLine);
    fclose(yyin);

}

int yywrap(){
    return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [ ]:
%%writefile program.txt

This is a sample file.

Writing program.txt


In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt

Word Count: 5
Character Count: 18
Space Count: 4
Line Count: 2


## Lex and Yacc combined

In [ ]:
#@title Writing YACC program
%%writefile program.y

%{
    #include<stdio.h>
    #include<stdlib.h>
%}
%token DIGIT LETTER UND NL
%%
stmt: variable NL {printf("Valid Identifier\n");exit(0);}
variable: LETTER alphanumeric;
alphanumeric: LETTER alphanumeric
            | DIGIT alphanumeric
            | UND alphanumeric
            | LETTER
            | DIGIT
            | UND;
%%

int yyerror(){
    printf("Invalid Identifier\n");
    exit(0);
}

void main(){
    printf("Enter the variable name: ");
    yyparse();
}

Overwriting program.y


In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include "y.tab.h"
%}
%%
[a-zA-Z] {return LETTER;}
[0-9] {return DIGIT;}
[_] {return UND;}
\n {return NL;}
. {return yytext[0];}
%%

In [2]:
%%writefile kkr.c
#include <stdio.h>
#include <string.h>
#include <ctype.h>

#define MAX 20

char prod[MAX][20];
char first[26][20], follow[26][20];
int n;

void add(char *s, char c)
{
    if (!strchr(s, c)) {
        int l = strlen(s);
        s[l] = c;
        s[l + 1] = '\0';
    }
}

int isNT(char c)
{
    return (c >= 'A' && c <= 'Z');
}

void FIRST(char nt)
{
    int i, j;
    for (i = 0; i < n; i++) {
        if (prod[i][0] != nt)
            continue;

        char *rhs = prod[i] + 2;

        if (rhs[0] == '#') {
            add(first[nt - 'A'], '#');
        }
        else if (!isNT(rhs[0])) {
            add(first[nt - 'A'], rhs[0]);
        }
        else {
            char x = rhs[0];

            if (first[x - 'A'][0] == '\0')
                FIRST(x);

            for (j = 0; first[x - 'A'][j]; j++)
                add(first[nt - 'A'], first[x - 'A'][j]);
        }
    }
}

void FOLLOW()
{
    int i, j, k;

    add(follow[prod[0][0] - 'A'], '$');

    for (k = 0; k < n; k++) {
        for (i = 0; i < n; i++) {
            char A = prod[i][0];
            char *rhs = prod[i] + 2;

            for (j = 0; rhs[j]; j++) {
                if (isNT(rhs[j])) {
                    char B = rhs[j];

                    if (rhs[j + 1] != '\0') {
                        char x = rhs[j + 1];

                        if (!isNT(x))
                            add(follow[B - 'A'], x);
                        else {
                            for (int m = 0; first[x - 'A'][m]; m++)
                                if (first[x - 'A'][m] != '#')
                                    add(follow[B - 'A'],
                                        first[x - 'A'][m]);
                        }
                    }
                    else {
                        for (int m = 0; follow[A - 'A'][m]; m++)
                            add(follow[B - 'A'],
                                follow[A - 'A'][m]);
                    }
                }
            }
        }
    }
}

int main()
{
    int i, j;

    printf("Enter number of productions: ");
    scanf("%d", &n);

    printf("Enter productions (use # for epsilon):\n");

    for (i = 0; i < n; i++)
        scanf("%s", prod[i]);

    /* FIRST */
    for (i = 0; i < n; i++)
        FIRST(prod[i][0]);

    /* FOLLOW */
    FOLLOW();

    printf("\nFIRST:\n");
    for (i = 0; i < n; i++) {
        char A = prod[i][0];

        if (i == 0 || prod[i - 1][0] != A)
            printf("FIRST(%c) = {%s}\n",
                   A, first[A - 'A']);
    }

    printf("\nFOLLOW:\n");
    for (i = 0; i < n; i++) {
        char A = prod[i][0];

        if (i == 0 || prod[i - 1][0] != A)
            printf("FOLLOW(%c) = {%s}\n",
                   A, follow[A - 'A']);
    }

    /* LL(1) Parsing Table */
    printf("\nLL(1) PARSING TABLE\n");
    printf("-------------------\n");

    for (i = 0; i < n; i++) {
        char A = prod[i][0];

        if (i > 0 && prod[i - 1][0] == A)
            continue;

        printf("\n%c : ", A);

        for (j = 0; j < n; j++) {
            if (prod[j][0] != A)
                continue;

            char *rhs = prod[j] + 2;

            if (rhs[0] == '#') {
                for (int k = 0; follow[A - 'A'][k]; k++)
                    printf("[%c,# -> #] ",
                           follow[A - 'A'][k]);
            }
            else if (!isNT(rhs[0])) {
                printf("[%c,%c -> %s] ",
                       A, rhs[0], rhs);
            }
            else {
                char B = rhs[0];

                for (int k = 0; first[B - 'A'][k]; k++)
                    if (first[B - 'A'][k] != '#')
                        printf("[%c,%c -> %s] ",
                               A, first[B - 'A'][k], rhs);
            }
        }
    }

    printf("\n");
    return 0;
}

Writing kkr.c


In [4]:
%%shell
gcc kkr.c
./a.out

Enter number of productions: 4
Enter productions (use # for epsilon):
3
3
3
2

FIRST:
FIRST(3) = {}
FIRST(2) = {}

FOLLOW:
FOLLOW(3) = {$}
FOLLOW(2) = {}

LL(1) PARSING TABLE
-------------------

3 : [3,  -> ] [3,  -> ] [3,  -> ] 
2 : [2,  -> ] 


if you want to use at txt as an input

In [ ]:
%%writefile program.txt

This is a sample file.

In [1]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out

/bin/bash: line 2: yacc: command not found
/bin/bash: line 3: lex: command not found
cc1: fatal error: y.tab.c: No such file or directory
compilation terminated.
cc1: fatal error: lex.yy.c: No such file or directory
compilation terminated.
/bin/bash: line 5: ./a.out: No such file or directory


CalledProcessError: Command '
yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out
' returned non-zero exit status 127.